# CosQA E5 + HyDE + re-ranking comparison

This notebook is the ordered entry point for AZH-517. It evaluates the four controlled systems in the project matrix: `e5`, `e5_hyde`, `e5_rerank`, and `e5_hyde_rerank`. Every system uses the pinned `CoIR-Retrieval/cosqa` data, text-only E5 prefixes, the same exact first-stage candidate depth, the same qrels, and the official COIR evaluator at `nDCG@10`. The combined row uses an E5-anchored weighted reciprocal-rank fusion of the original E5, HyDE, and both reranker rankings; its fixed `rrf_k` and weights are recorded in the result artifact.

TQE is implemented as HyDE with a local `google/flan-t5-base` generator. The cross-encoder only reorders the IDs returned by the E5 first stage. The default is a real-model smoke run and is not benchmark evidence; set `E5_HYDE_RERANK_MODE=benchmark` for the complete declared split and corpus. `E5_HYDE_RERANK_CANDIDATE_DEPTH` makes candidate depth explicit (default `1000`); set it to `10` to match the saved E5 baseline benchmark.

No generated text, result score, or benchmark claim is manually entered in this notebook. A dependency, model, data, or evaluator failure is persisted as a `blocked` artifact.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys

workspace_root = Path.cwd()
while workspace_root != workspace_root.parent and not (workspace_root / 'code-retrieval' / 'src').exists():
    workspace_root = workspace_root.parent
project_dir = workspace_root / 'code-retrieval'
if not (project_dir / 'src').exists():
    project_dir = Path.cwd()
    workspace_root = project_dir.parent
os.chdir(project_dir)
sys.path.insert(0, str(project_dir / 'src'))

from e5_baseline import environment_metadata, package_versions, set_seed
from e5_hyde_rerank import (
    ExperimentConfig,
    blocked_result,
    load_cosqa,
    run_experiment,
    select_run_data,
    write_comparison_artifacts,
)

print('Project directory:', project_dir.resolve())
print('Python:', sys.version.split()[0])

## 1. Environment and explicit controls

The generator, re-ranker, E5 settings, dataset revision, seed, candidate depth, and execution mode are configuration, not hidden notebook state.

In [ ]:
RUN_MODE = os.environ.get('E5_HYDE_RERANK_MODE', 'smoke').strip().lower()
if RUN_MODE not in {'smoke', 'benchmark'}:
    raise ValueError('E5_HYDE_RERANK_MODE must be smoke or benchmark')

batch_size = int(os.environ.get('E5_HYDE_RERANK_BATCH_SIZE', '4' if RUN_MODE == 'smoke' else '128'))
candidate_depth = int(os.environ.get('E5_HYDE_RERANK_CANDIDATE_DEPTH', '1000'))
if candidate_depth <= 0:
    raise ValueError('E5_HYDE_RERANK_CANDIDATE_DEPTH must be positive')
torch_threads = int(os.environ.get('E5_HYDE_RERANK_TORCH_THREADS', '0'))
if torch_threads > 0:
    import torch
    torch.set_num_threads(torch_threads)

config = ExperimentConfig(
    run_mode=RUN_MODE,
    batch_size=batch_size,
    candidate_depth=candidate_depth,
    reranker_batch_size=batch_size,
    cache_dir='artifacts/e5_hyde_rerank/cache',
    artifact_dir='artifacts/e5_hyde_rerank',
)
set_seed(config.seed)
print(json.dumps(config.as_dict(), indent=2, sort_keys=True))
print('Installed packages:')
print(json.dumps(package_versions(['coir-eval', 'datasets', 'faiss-cpu', 'numpy', 'pytrec-eval-terrier', 'sentence-transformers', 'torch', 'transformers']), indent=2, sort_keys=True))
print('Hardware/runtime:')
print(json.dumps(environment_metadata(config, repo_root=workspace_root), indent=2, sort_keys=True, default=str))

## 2. Load and inspect the real CosQA schema

The loader reads the separate corpus, query, and test-qrels configurations from one pinned dataset revision before any model call. Titles and unrelated fields are not passed into E5 or the cross-encoder.

In [ ]:
data = load_cosqa(config)
print(json.dumps(data.schema, indent=2, sort_keys=True, default=str))
print({
    'corpus_count': len(data.corpus),
    'query_count_with_test_qrels': len(data.queries),
    'qrels_query_count': len(data.qrels),
    'qrels_judgment_count': sum(len(rels) for rels in data.qrels.values()),
})
print('Corpus example:', next(iter(data.corpus.items())))
print('Query example:', next(iter(data.queries.items())))
print('Qrels example:', next(iter(data.qrels.items())))

## 3. Select the evidence level

Smoke mode uses a small labeled subset and records exclusions. Benchmark mode must use every declared test-qrels query and every corpus document.

In [ ]:
run_data = select_run_data(data, config)
if config.run_mode == 'benchmark' and (len(run_data.queries) != len(data.queries) or len(run_data.corpus) != len(data.corpus)):
    raise RuntimeError('benchmark mode must use the complete declared CosQA test queries and corpus')
print(json.dumps({
    'run_mode': config.run_mode,
    'query_count': len(run_data.queries),
    'corpus_count': len(run_data.corpus),
    'qrels_query_count': len(run_data.qrels),
    'qrels_judgment_count': sum(len(rels) for rels in run_data.qrels.values()),
    'exclusions': run_data.exclusions,
}, indent=2))

## 4. Run the controlled four-system experiment

The reusable runner loads real E5, HyDE, and cross-encoder models, resolves their revisions, validates each cache identity, and evaluates all systems against the same qrels. The original query remains the evaluation query; only the model input representation changes for the HyDE systems. Re-ranking is scored in one batched pass per candidate collection, and the final combined ranking is a deterministic weighted reciprocal-rank fusion that retains the configured 1,000-document depth.

In [ ]:
notebook_path = project_dir / 'notebooks' / 'e5_hyde_rerank_experiment.ipynb'
notebook_sha256 = hashlib.sha256(notebook_path.read_bytes()).hexdigest() if notebook_path.exists() else None
environment = environment_metadata(config, repo_root=workspace_root)

try:
    result, artifact_paths = run_experiment(
        config,
        repo_root=workspace_root,
        notebook_sha256=notebook_sha256,
    )
except Exception as error:
    blocker = f'{type(error).__name__}: {error}'
    result = blocked_result(
        config,
        blocker=blocker,
        environment=environment,
        repo_root=workspace_root,
        notebook_sha256=notebook_sha256,
    )
    artifact_paths = write_comparison_artifacts(config, result)
    print('Execution blocker recorded:', blocker)

print(json.dumps({
    'status': result['status'],
    'benchmark_evidence': result['benchmark_evidence'],
    'system_ids': result['system_ids'],
    'artifact_paths': artifact_paths,
}, indent=2))

## 5. Inspect the comparison and interpretation boundary

The table is produced from evaluator output, not entered by hand. Deltas are absolute changes versus the same-run E5 row. Smoke values prove wiring only, and blocked values are null rather than partial benchmark evidence.

In [ ]:
print(json.dumps(result['results'], indent=2, sort_keys=True))
print(json.dumps({
    'metric': result['metric'],
    'status': result['status'],
    'benchmark_evidence': result['benchmark_evidence'],
    'candidate_depth': result['candidate_depth'],
    'dataset_revision': result['dataset']['revision'],
    'hyde': result['hyde'],
    'reranker': result['reranker'],
    'exclusions': result['exclusions'],
    'limitations': result['limitations'],
    'provenance': result['artifact_provenance'],
}, indent=2, sort_keys=True, default=str))